# Research Study: End-to-End Voice Deepfake Detection with Squeeze-and-Excitation ResNet-18
## ASVspoof 2019 Logical Access (LA) Benchmark: Full Tri-Partition Evaluation (Train, Dev, and Eval)

### Abstract and Scientific Motivation
Deepfake speech generation algorithms (including autoregressive neural acoustic models, diffusion speech synthesizers, and neural vocoders such as WaveNet, HiFi-GAN, and Parallel WaveGAN) induce subtle spectral artifacts, harmonic distortions, and phase irregularities. While shallow neural models frequently memorize specific artifacts present in known training attacks, they suffer catastrophic performance degradation when deployed against unseen generative architectures.

To solve this critical generalization bottleneck, this investigation formulates **SE-ResNet-18 (Squeeze-and-Excitation Residual Network)** optimized for end-to-end voice anti-spoofing. The architecture integrates:
1. **GPU-Accelerated Log-Mel Front-End**: High-resolution 80-bin Mel filterbank extraction with dynamic range compression and instance normalization executed directly on GPU tensor streams.
2. **Channel-Wise Attention via Squeeze-and-Excitation (SE)**: Squeeze operation (global spatial average pooling) coupled with an excitation mechanism (two-layer bottleneck MLP with ReLU and Sigmoid gating) that dynamically recalibrates spectro-temporal feature maps:
$$\mathbf{s} = \sigma(\mathbf{W}_2 \delta(\mathbf{W}_1 \mathbf{z}))$$
$$\widetilde{\mathbf{X}}_c = s_c \cdot \mathbf{X}_c$$
3. **Focal Loss with Label Smoothing**: Counteracts the severe 1:9 class imbalance between authentic human speech and synthetic attacks while preventing overconfident calibration errors:
$$\mathcal{L}_{\text{Focal}} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$
4. **Complete Tri-Partition Evaluation Protocol**: Unlike conventional closed-world experiments that restrict testing to the Development partition (A01-A06), this study executes exhaustive batch evaluation over all **71,237 utterances of the official ASVspoof 2019 Evaluation partition**, rigorously quantifying out-of-distribution generalization across thirteen distinct attack algorithms (**A07 through A19**).

### Execution Architecture
This notebook executes a fully automated, standalone end-to-end research pipeline in a single Kaggle session:
- Cell 1: Hardware diagnostics and deterministic random seed initialization
- Cell 2: Multi-path dataset discovery across standard Kaggle input topologies
- Cell 3: Unified tri-partition protocol parsing (Train, Dev, Eval - 121,461 utterances)
- Cell 4: Speaker independence audit and metadata verification
- Cells 5-11: Exhaustive Exploratory Data Analysis (EDA) suite producing 7 publication figures
- Cell 12: Data augmentation pipeline (SpecAugment, time masking, frequency masking)
- Cell 13: Memory-efficient PyTorch Dataset with WeightedRandomSampler
- Cell 14: SE-ResNet-18 architecture with Squeeze-and-Excitation blocks and latent projection
- Cell 15: Focal Loss formulation with label smoothing and Cosine Annealing scheduler
- Cell 16: Biometric evaluation engine (EER, min t-DCF, ROC, DET, PR)
- Cell 17: Full 20-epoch training and validation tracking with real-time Dev EER
- Cell 18: Training diagnostic curves (Loss trajectories, EER convergence)
- Cell 19: Full-scale Evaluation Partition batch inference across all 71,237 utterances
- Cells 20-22: Biometric performance curves (ROC, DET, PR on Dev and Eval)
- Cell 23: Normalized Confusion Matrix on the Evaluation partition
- Cell 24: Granular attack-by-attack vulnerability audit (A01 through A19)
- Cell 25: 2D t-SNE latent representation clustering (Bonafide vs Known vs Unseen attacks)
- Cell 26: Spectro-temporal explainability via Grad-CAM saliency heatmaps
- Cell 27: Live single-file inference demonstration
- Cell 28: Final artifact inventory and executive performance summary


In [ ]:
import os
import sys
import time
import math
import json
import random
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, precision_recall_curve, average_precision_score
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchaudio

warnings.filterwarnings("ignore")

seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()

print("Hardware and Runtime Diagnostics:")
print(f"  PyTorch Version:  {torch.__version__}")
print(f"  Torchaudio:       {torchaudio.__version__}")
print(f"  Compute Device:   {device}")
if torch.cuda.is_available():
    print(f"  GPU Identifier:   {torch.cuda.get_device_name(0)}")
    print(f"  VRAM Allocated:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"  Mixed Precision:  Enabled (torch.amp.autocast)")
else:
    print("  WARNING: GPU accelerator not detected. Execution will proceed on CPU.")

fig_dir = "/kaggle/working/figures"
weights_dir = "/kaggle/working/models"
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(weights_dir, exist_ok=True)
print(f"Output Figure Directory:  {fig_dir}")
print(f"Output Model Directory:   {weights_dir}")


In [ ]:
def locate_dataset():
    search_roots = ["/kaggle/input", "/kaggle/working", "."]
    candidates = [
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA",
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA",
        "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset",
        "/kaggle/input/asvpoof-2019-dataset/LA/LA",
        "/kaggle/input/asvpoof-2019-dataset/LA",
        "/kaggle/input/asvpoof-2019-dataset",
        "/kaggle/input/asvspoof-2019-dataset/LA/LA",
        "/kaggle/input/asvspoof-2019-dataset/LA",
        "/kaggle/input/asvspoof-2019-dataset",
        "/kaggle/input/asvspoof-2019/LA/LA",
        "/kaggle/input/asvspoof-2019/LA",
        "/kaggle/input/asvspoof-2019",
        "/kaggle/input/asvspoof2019/LA/LA",
        "/kaggle/input/asvspoof2019/LA",
        "/kaggle/input/asvspoof2019"
    ]
    protocols = {"train": None, "dev": None, "eval": None}
    audio_dirs = {"train": None, "dev": None, "eval": None}

    for c in candidates:
        if os.path.isdir(c):
            for part, sfx in [("train", "trn"), ("dev", "trl"), ("eval", "trl")]:
                proto_p = os.path.join(c, "ASVspoof2019_LA_cm_protocols", f"ASVspoof2019.LA.cm.{part}.{sfx}.txt")
                if os.path.isfile(proto_p) and not protocols[part]:
                    protocols[part] = proto_p
                flac_p = os.path.join(c, f"ASVspoof2019_LA_{part}", "flac")
                if os.path.isdir(flac_p) and not audio_dirs[part]:
                    audio_dirs[part] = flac_p
                elif os.path.isdir(os.path.join(c, f"ASVspoof2019_LA_{part}")) and not audio_dirs[part]:
                    audio_dirs[part] = os.path.join(c, f"ASVspoof2019_LA_{part}")

    if any(v is None for v in list(protocols.values()) + list(audio_dirs.values())):
        for s_root in search_roots:
            if not os.path.exists(s_root):
                continue
            for root, dirs, files in os.walk(s_root, followlinks=True):
                for f in files:
                    fl = f.lower()
                    if "cm" in fl and fl.endswith(".txt") and not f.startswith("._"):
                        if "train" in fl and ("trn" in fl or "train" in fl) and not protocols["train"]:
                            protocols["train"] = os.path.join(root, f)
                        elif "dev" in fl and ("trl" in fl or "dev" in fl) and not protocols["dev"]:
                            protocols["dev"] = os.path.join(root, f)
                        elif "eval" in fl and ("trl" in fl or "eval" in fl) and not protocols["eval"]:
                            protocols["eval"] = os.path.join(root, f)
                for d in list(dirs):
                    dl = d.lower()
                    for part in ["train", "dev", "eval"]:
                        if (f"la_{part}" in dl or f"la.{part}" in dl or f"_{part}" in dl) and not audio_dirs[part]:
                            sub_flac = os.path.join(root, d, "flac")
                            if os.path.isdir(sub_flac):
                                audio_dirs[part] = sub_flac
                            elif os.path.isdir(os.path.join(root, d)):
                                audio_dirs[part] = os.path.join(root, d)
                if "flac" in dirs:
                    dirs.remove("flac")

    return protocols, audio_dirs

protocols, audio_dirs = locate_dataset()

print("Resolved Dataset Partitions:")
for p in ["train", "dev", "eval"]:
    proto_stat = "FOUND" if protocols[p] and os.path.isfile(protocols[p]) else "MISSING"
    audio_stat = "FOUND" if audio_dirs[p] and os.path.isdir(audio_dirs[p]) else "MISSING"
    n_files = len(os.listdir(audio_dirs[p])) if audio_stat == "FOUND" else 0
    print(f"  [{p.upper()}]")
    print(f"    Protocol:  {proto_stat} -> {protocols[p]}")
    print(f"    Audio Dir: {audio_stat} -> {audio_dirs[p]} ({n_files:,} files)")

if not protocols["train"] or not os.path.isfile(protocols["train"]):
    raise FileNotFoundError("Critical: Train protocol file missing. Ensure ASVspoof 2019 dataset is attached to Kaggle notebook.")
if not audio_dirs["train"] or not os.path.isdir(audio_dirs["train"]):
    raise FileNotFoundError("Critical: Train audio directory missing.")


In [ ]:
parsed_records = []

for partition in ["train", "dev", "eval"]:
    proto_path = protocols.get(partition)
    audio_path = audio_dirs.get(partition)
    if not proto_path or not os.path.isfile(proto_path):
        print(f"Warning: Partition {partition} protocol file not accessible.")
        continue
    if not audio_path or not os.path.isdir(audio_path):
        print(f"Warning: Partition {partition} audio directory not accessible.")
        continue

    with open(proto_path, "r", encoding="utf-8") as f:
        for line in f:
            tokens = line.strip().split()
            if len(tokens) < 5:
                continue
            spk_id = tokens[0]
            audio_id = tokens[1]
            env_id = tokens[2]
            atk_token = tokens[3]
            key_token = tokens[4]

            atk_id = "Bonafide" if key_token == "bonafide" or atk_token == "-" else atk_token
            is_spoof = 1 if key_token == "spoof" else 0

            file_full_path = os.path.join(audio_path, f"{audio_id}.flac")
            parsed_records.append({
                "speaker_id": spk_id,
                "audio_id": audio_id,
                "environment_id": env_id,
                "attack_id": atk_id,
                "key": key_token,
                "is_spoof": is_spoof,
                "partition": partition,
                "file_path": file_full_path
            })

manifest_df = pd.DataFrame(parsed_records)
print(f"Total Database Utterances Parsed: {len(manifest_df):,}")

partition_summary = []
for p in ["train", "dev", "eval"]:
    sub = manifest_df[manifest_df["partition"] == p]
    if len(sub) == 0:
        continue
    bon_cnt = int((sub["key"] == "bonafide").sum())
    spf_cnt = int((sub["key"] == "spoof").sum())
    tot_cnt = len(sub)
    ratio_str = f"{spf_cnt / max(bon_cnt, 1):.2f}:1"
    attacks_present = sorted([a for a in sub["attack_id"].unique() if a != "Bonafide"])
    partition_summary.append({
        "Partition": p.upper(),
        "Total Utterances": f"{tot_cnt:,}",
        "Bonafide": f"{bon_cnt:,}",
        "Spoof": f"{spf_cnt:,}",
        "Spoof:Bonafide Ratio": ratio_str,
        "Unique Speakers": sub["speaker_id"].nunique(),
        "Attack IDs": ", ".join(attacks_present)
    })

summary_display = pd.DataFrame(partition_summary)
print(summary_display.to_string(index=False))

train_df = manifest_df[manifest_df["partition"] == "train"].reset_index(drop=True)
dev_df = manifest_df[manifest_df["partition"] == "dev"].reset_index(drop=True)
eval_df = manifest_df[manifest_df["partition"] == "eval"].reset_index(drop=True)

print("")
print(f"Partition Split Complete:")
print(f"  Train: {len(train_df):,} utterances (A01-A06)")
print(f"  Dev:   {len(dev_df):,} utterances (A01-A06)")
print(f"  Eval:  {len(eval_df):,} utterances (A07-A19 unseen attacks)")


In [ ]:
spks_train = set(train_df["speaker_id"].unique())
spks_dev = set(dev_df["speaker_id"].unique())
spks_eval = set(eval_df["speaker_id"].unique()) if len(eval_df) > 0 else set()

ov_train_dev = spks_train.intersection(spks_dev)
ov_train_eval = spks_train.intersection(spks_eval)
ov_dev_eval = spks_dev.intersection(spks_eval)

print("Speaker Independence Audit:")
print(f"  Train Unique Speakers: {len(spks_train)}")
print(f"  Dev Unique Speakers:   {len(spks_dev)}")
print(f"  Eval Unique Speakers:  {len(spks_eval)}")
print(f"  Overlap (Train & Dev):  {len(ov_train_dev)} (Expected: 0)")
print(f"  Overlap (Train & Eval): {len(ov_train_eval)} (Expected: 0)")
print(f"  Overlap (Dev & Eval):   {len(ov_dev_eval)} (Expected: 0)")

assert len(ov_train_dev) == 0, "Data leakage detected: Speaker overlap between Train and Dev partitions."
assert len(ov_train_eval) == 0, "Data leakage detected: Speaker overlap between Train and Eval partitions."
print("Speaker independence strictly verified across all three experimental partitions.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

parts = [p for p in ["train", "dev", "eval"] if len(manifest_df[manifest_df["partition"] == p]) > 0]
bon_counts = [int(((manifest_df["partition"] == p) & (manifest_df["key"] == "bonafide")).sum()) for p in parts]
spf_counts = [int(((manifest_df["partition"] == p) & (manifest_df["key"] == "spoof")).sum()) for p in parts]

x_idx = np.arange(len(parts))
b_width = 0.35

axes[0].bar(x_idx - b_width/2, bon_counts, b_width, label="Authentic (Bonafide)", color="steelblue")
axes[0].bar(x_idx + b_width/2, spf_counts, b_width, label="Synthetic (Spoof)", color="firebrick")
axes[0].set_title("Class Utterance Breakdown Across Partitions", fontsize=12)
axes[0].set_xlabel("Dataset Partition", fontsize=11)
axes[0].set_ylabel("Total Number of Utterances", fontsize=11)
axes[0].set_xticks(x_idx)
axes[0].set_xticklabels([p.upper() for p in parts], fontsize=10)
axes[0].legend(fontsize=10)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

for i in range(len(parts)):
    axes[0].text(x_idx[i] - b_width/2, bon_counts[i] + max(spf_counts)*0.015, f"{bon_counts[i]:,}", ha="center", fontsize=9)
    axes[0].text(x_idx[i] + b_width/2, spf_counts[i] + max(spf_counts)*0.015, f"{spf_counts[i]:,}", ha="center", fontsize=9)

ratios = [spf_counts[i] / max(bon_counts[i], 1) for i in range(len(parts))]
axes[1].bar([p.upper() for p in parts], ratios, color="darkslateblue", width=0.4)
axes[1].set_title("Class Imbalance Ratio (Spoof : Bonafide)", fontsize=12)
axes[1].set_xlabel("Dataset Partition", fontsize=11)
axes[1].set_ylabel("Imbalance Ratio", fontsize=11)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

for i, r in enumerate(ratios):
    axes[1].text(i, r + 0.15, f"{r:.2f}:1", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "01_class_distribution_breakdown.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 01: Class distribution breakdown.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

train_dev_atks = manifest_df[manifest_df["partition"].isin(["train", "dev"]) & (manifest_df["attack_id"] != "Bonafide")]
td_counts = train_dev_atks["attack_id"].value_counts().sort_index()

eval_atks = manifest_df[(manifest_df["partition"] == "eval") & (manifest_df["attack_id"] != "Bonafide")]
eval_counts = eval_atks["attack_id"].value_counts().sort_index()

axes[0].bar(td_counts.index, td_counts.values, color="coral", edgecolor="black", width=0.55)
axes[0].set_title("Known Attack Distribution: Train & Dev (A01 - A06)", fontsize=12)
axes[0].set_xlabel("Spoofing Attack Algorithm ID", fontsize=11)
axes[0].set_ylabel("Number of Utterances", fontsize=11)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)
for idx, val in enumerate(td_counts.values):
    axes[0].text(idx, val + max(td_counts.values)*0.015, f"{val:,}", ha="center", fontsize=9)

axes[1].bar(eval_counts.index, eval_counts.values, color="mediumpurple", edgecolor="black", width=0.6)
axes[1].set_title("Unseen Out-of-Distribution Attacks: Evaluation (A07 - A19)", fontsize=12)
axes[1].set_xlabel("Spoofing Attack Algorithm ID", fontsize=11)
axes[1].set_ylabel("Number of Utterances", fontsize=11)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
for idx, val in enumerate(eval_counts.values):
    axes[1].text(idx, val + max(eval_counts.values)*0.015, f"{val:,}", ha="center", fontsize=8, rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "02_attack_distribution_analysis.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 02: Attack distribution analysis.")


In [ ]:
sample_rows = manifest_df.sample(min(200, len(manifest_df)), random_state=42)
durations = []
sample_rates = []

for _, r in sample_rows.iterrows():
    fpath = r["file_path"]
    if os.path.isfile(fpath):
        try:
            info = sf.info(fpath)
            durations.append(info.duration)
            sample_rates.append(info.samplerate)
        except Exception:
            pass

durations = np.array(durations)
sample_rates = np.array(sample_rates)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(durations, bins=25, color="teal", edgecolor="black", alpha=0.8)
axes[0].axvline(np.mean(durations), color="red", linestyle="--", lw=2, label=f"Mean: {np.mean(durations):.2f}s")
axes[0].axvline(4.0, color="orange", linestyle=":", lw=2, label="Fixed Target Window: 4.00s")
axes[0].set_title("Speech Duration Distribution (Audit Sample N=200)", fontsize=12)
axes[0].set_xlabel("Audio Duration (Seconds)", fontsize=11)
axes[0].set_ylabel("Utterance Count", fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, linestyle="--", alpha=0.3)

sr_counts = pd.Series(sample_rates).value_counts()
axes[1].bar([f"{int(sr)} Hz" for sr in sr_counts.index], sr_counts.values, color="steelblue", width=0.3)
axes[1].set_title("Sample Rate Consistency Audit", fontsize=12)
axes[1].set_xlabel("Sampling Frequency", fontsize=11)
axes[1].set_ylabel("Utterance Count", fontsize=11)
axes[1].grid(axis="y", linestyle="--", alpha=0.3)
for i, v in enumerate(sr_counts.values):
    axes[1].text(i, v + 2, f"{v} (100%)", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "03_audio_duration_length_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 03: Audio duration and sample rate audit.")


In [ ]:
def read_audio_file(file_path):
    sig, sr = sf.read(file_path, dtype="float32")
    if sig.ndim > 1:
        sig = np.mean(sig, axis=1)
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
        sig = resampler(torch.from_numpy(sig)).numpy()
        sr = 16000
    return sig, sr

bon_sample_path = train_df[train_df["key"] == "bonafide"].iloc[0]["file_path"]
a01_sample_path = train_df[train_df["attack_id"] == "A01"].iloc[0]["file_path"]
a04_sample_path = train_df[train_df["attack_id"] == "A04"].iloc[0]["file_path"]

sig_bon, sr = read_audio_file(bon_sample_path)
sig_a01, _ = read_audio_file(a01_sample_path)
sig_a04, _ = read_audio_file(a04_sample_path)

fig, axes = plt.subplots(3, 2, figsize=(15, 8), sharex="col")

zoom_len = 1600

signals = [
    (sig_bon, "Authentic Human Voice (VCTK Corpus)", "steelblue"),
    (sig_a01, "TTS: Neural Acoustic + WaveNet (A01)", "firebrick"),
    (sig_a04, "Voice Conversion: Pitch Shifting + STRAIGHT (A04)", "darkorange")
]

for row_idx, (sig, label, col) in enumerate(signals):
    t_full = np.linspace(0, len(sig) / sr, len(sig))
    axes[row_idx, 0].plot(t_full, sig, color=col, lw=0.6)
    axes[row_idx, 0].set_ylabel("Amplitude", fontsize=10)
    axes[row_idx, 0].set_title(f"Full Utterance Waveform: {label}", fontsize=11)
    axes[row_idx, 0].grid(True, linestyle="--", alpha=0.3)

    t_zoom = np.linspace(0, zoom_len / sr, zoom_len)
    start_pt = min(int(sr * 0.8), len(sig) - zoom_len)
    axes[row_idx, 1].plot(t_zoom * 1000, sig[start_pt:start_pt + zoom_len], color=col, lw=1.2)
    axes[row_idx, 1].set_ylabel("Amplitude", fontsize=10)
    axes[row_idx, 1].set_title(f"Zoomed Glottal Waveform (100ms): {label}", fontsize=11)
    axes[row_idx, 1].grid(True, linestyle="--", alpha=0.3)

axes[2, 0].set_xlabel("Time (Seconds)", fontsize=11)
axes[2, 1].set_xlabel("Time (Milliseconds)", fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "04_waveform_time_domain_comparison.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 04: Waveform and glottal pulse inspection.")


In [ ]:
from scipy import signal as scipy_signal

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for sig, label, col in signals:
    freqs, psd = scipy_signal.welch(sig, fs=sr, nperseg=1024)
    psd_db = 10 * np.log10(psd + 1e-12)
    axes[0].plot(freqs, psd_db, label=label, color=col, lw=1.5)

axes[0].set_title("Power Spectral Density (Before Preprocessing)", fontsize=12)
axes[0].set_xlabel("Frequency (Hz)", fontsize=11)
axes[0].set_ylabel("Power / Frequency (dB/Hz)", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle="--", alpha=0.3)
axes[0].set_xlim(0, 8000)

def normalize_waveform(sig, target_len=64000):
    if len(sig) < target_len:
        rep = math.ceil(target_len / len(sig))
        sig = np.tile(sig, rep)[:target_len]
    else:
        start = (len(sig) - target_len) // 2
        sig = sig[start:start + target_len]
    sig = sig - np.mean(sig)
    peak = np.max(np.abs(sig))
    if peak > 1e-6:
        sig = sig / peak
    return sig

for sig, label, col in signals:
    sig_proc = normalize_waveform(sig)
    freqs, psd = scipy_signal.welch(sig_proc, fs=sr, nperseg=1024)
    psd_db = 10 * np.log10(psd + 1e-12)
    axes[1].plot(freqs, psd_db, label=label, color=col, lw=1.5)

axes[1].set_title("Power Spectral Density (After Length Normalization & Peak Scaling)", fontsize=12)
axes[1].set_xlabel("Frequency (Hz)", fontsize=11)
axes[1].set_ylabel("Normalized Power (dB/Hz)", fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle="--", alpha=0.3)
axes[1].set_xlim(0, 8000)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "05_power_spectral_density_frequency_rolloff.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 05: Power spectral density and frequency rolloff.")


In [ ]:
mel_extractor_demo = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=512,
    win_length=400,
    hop_length=160,
    f_min=20,
    f_max=8000,
    n_mels=80
)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for idx, (sig, label, _) in enumerate(signals):
    sig_proc = normalize_waveform(sig)
    tens = torch.from_numpy(sig_proc).float().unsqueeze(0)
    mel_spec = mel_extractor_demo(tens)
    log_mel = torch.log(mel_spec + 1e-6).squeeze(0).numpy()

    im = axes[idx].imshow(log_mel, origin="lower", aspect="auto", cmap="viridis", interpolation="nearest")
    axes[idx].set_title(f"80-Bin Log-Mel Spectrogram: {label}", fontsize=11)
    axes[idx].set_ylabel("Mel Frequency Bins", fontsize=10)
    fig.colorbar(im, ax=axes[idx], format="%+2.0f dB")

axes[2].set_xlabel("Time Frames (10ms Hop)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "06_log_mel_spectrogram_representations.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 06: Log-Mel spectrogram representations across attacks.")


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=False)

raw_sig = sig_bon
axes[0].plot(np.linspace(0, len(raw_sig)/16000, len(raw_sig)), raw_sig, color="black", lw=0.6)
axes[0].set_title("Step 1: Raw Dynamic-Length Audio Signal (FLAC 16 kHz)", fontsize=11)
axes[0].set_ylabel("Amplitude", fontsize=9)
axes[0].grid(True, linestyle="--", alpha=0.3)

norm_sig = normalize_waveform(raw_sig, target_len=64000)
axes[1].plot(np.linspace(0, 4.0, 64000), norm_sig, color="teal", lw=0.6)
axes[1].set_title("Step 2: Time-Domain Truncation / Circular Padding to Fixed 4.00s Window (64,000 Samples)", fontsize=11)
axes[1].set_ylabel("Normalized Amp", fontsize=9)
axes[1].grid(True, linestyle="--", alpha=0.3)

mel_spec = mel_extractor_demo(torch.from_numpy(norm_sig).float().unsqueeze(0))
log_mel = torch.log(mel_spec + 1e-6).squeeze(0).numpy()
axes[2].imshow(log_mel, origin="lower", aspect="auto", cmap="magma")
axes[2].set_title("Step 3: GPU STFT and 80-Channel Log-Mel Filterbank Projection", fontsize=11)
axes[2].set_ylabel("Mel Bins (0-80)", fontsize=9)

mean_val = np.mean(log_mel)
std_val = np.std(log_mel) + 1e-6
inst_norm_mel = (log_mel - mean_val) / std_val
axes[3].imshow(inst_norm_mel, origin="lower", aspect="auto", cmap="magma")
axes[3].set_title("Step 4: Instance Normalization (Zero Mean, Unit Variance per Utterance)", fontsize=11)
axes[3].set_ylabel("Mel Bins (0-80)", fontsize=9)
axes[3].set_xlabel("Time Frame Index (401 Frames @ 160 Hop Length)", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "07_audio_preprocessing_pipeline.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 07: Audio preprocessing transformation pipeline.")


In [ ]:
class SpecAugmentModule(nn.Module):
    def __init__(self, freq_mask_max=8, time_mask_max=24, num_freq_masks=2, num_time_masks=2):
        super().__init__()
        self.freq_mask_max = freq_mask_max
        self.time_mask_max = time_mask_max
        self.num_freq_masks = num_freq_masks
        self.num_time_masks = num_time_masks

    def forward(self, mel_spec):
        if not self.training:
            return mel_spec
        cloned = mel_spec.clone()
        b, c, f, t = cloned.shape
        for _ in range(self.num_freq_masks):
            f_len = random.randint(0, self.freq_mask_max)
            f_zero = random.randint(0, max(0, f - f_len))
            cloned[:, :, f_zero:f_zero + f_len, :] = 0.0
        for _ in range(self.num_time_masks):
            t_len = random.randint(0, self.time_mask_max)
            t_zero = random.randint(0, max(0, t - t_len))
            cloned[:, :, :, t_zero:t_zero + t_len] = 0.0
        return cloned

print("SpecAugment time-frequency masking module initialized.")


In [ ]:
class ASVSpoofDataset(Dataset):
    def __init__(self, dataframe, target_len=64000, is_train=False):
        self.df = dataframe.reset_index(drop=True)
        self.target_len = target_len
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row["file_path"]
        label = int(row["is_spoof"])

        try:
            sig, sr = sf.read(file_path, dtype="float32")
            if sig.ndim > 1:
                sig = np.mean(sig, axis=1)
        except Exception:
            sig = np.zeros(self.target_len, dtype="float32")

        curr_len = len(sig)
        if curr_len < self.target_len:
            repeat_factor = math.ceil(self.target_len / max(curr_len, 1))
            sig = np.tile(sig, repeat_factor)[:self.target_len]
        elif curr_len > self.target_len:
            if self.is_train:
                max_start = curr_len - self.target_len
                start = random.randint(0, max_start)
                sig = sig[start:start + self.target_len]
            else:
                start = (curr_len - self.target_len) // 2
                sig = sig[start:start + self.target_len]

        if self.is_train and random.random() < 0.3:
            scale = random.uniform(0.7, 1.2)
            sig = sig * scale

        peak = np.max(np.abs(sig))
        if peak > 1e-6:
            sig = sig / peak

        return torch.from_numpy(sig).float(), label

train_dataset = ASVSpoofDataset(train_df, is_train=True)
dev_dataset = ASVSpoofDataset(dev_df, is_train=False)
eval_dataset = ASVSpoofDataset(eval_df, is_train=False) if len(eval_df) > 0 else None

train_targets = train_df["is_spoof"].values
bon_count = (train_targets == 0).sum()
spf_count = (train_targets == 1).sum()
class_weights = [1.0 / max(bon_count, 1), 1.0 / max(spf_count, 1)]
sample_weights = [class_weights[int(t)] for t in train_targets]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_dataset), replacement=True)

batch_size = 64
num_workers = 2

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=num_workers, pin_memory=True, drop_last=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
eval_loader = DataLoader(eval_dataset, batch_size=128, shuffle=False, num_workers=num_workers, pin_memory=True) if eval_dataset else None

print(f"DataLoaders Constructed:")
print(f"  Train: {len(train_loader)} batches (Batch Size: {batch_size}, Weighted Balanced Sampling)")
print(f"  Dev:   {len(dev_loader)} batches (Batch Size: {batch_size})")
if eval_loader:
    print(f"  Eval:  {len(eval_loader)} batches (Batch Size: 128, Full 71,237 Utterances)")


In [ ]:
class SELayer(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class SEBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.se = SELayer(planes, reduction=reduction)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class SEResNet18(nn.Module):
    def __init__(self, num_classes=2, latent_dim=128):
        super().__init__()
        self.mel_layer = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000,
            n_fft=512,
            win_length=400,
            hop_length=160,
            f_min=20,
            f_max=8000,
            n_mels=80
        )
        self.spec_augment = SpecAugmentModule(freq_mask_max=8, time_mask_max=24)

        self.in_planes = 64
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_latent = nn.Linear(512, latent_dim)
        self.bn_latent = nn.BatchNorm1d(latent_dim)
        self.fc_out = nn.Linear(latent_dim, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(SEBasicBlock(self.in_planes, planes, stride=s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def extract_features(self, raw_audio):
        with torch.no_grad():
            mel = self.mel_layer(raw_audio)
            log_mel = torch.log(mel + 1e-6).unsqueeze(1)
            mean = log_mel.mean(dim=[-2, -1], keepdim=True)
            std = log_mel.std(dim=[-2, -1], keepdim=True) + 1e-5
            norm_mel = (log_mel - mean) / std
        if self.training:
            norm_mel = self.spec_augment(norm_mel)
        return norm_mel

    def forward(self, raw_audio):
        x = self.extract_features(raw_audio)
        x = self.maxpool(F.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).flatten(1)
        latent = F.relu(self.bn_latent(self.fc_latent(x)))
        logits = self.fc_out(latent)
        return logits

    def extract_latent(self, raw_audio):
        x = self.extract_features(raw_audio)
        x = self.maxpool(F.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).flatten(1)
        latent = F.relu(self.bn_latent(self.fc_latent(x)))
        return latent

model = SEResNet18(num_classes=2, latent_dim=128).to(device)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SE-ResNet-18 Instantiated. Trainable Parameters: {param_count:,}")


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none", label_smoothing=self.label_smoothing)
        p_t = torch.exp(-ce)
        alpha_t = torch.where(targets == 1, self.alpha, 1.0 - self.alpha)
        loss = alpha_t * ((1.0 - p_t) ** self.gamma) * ce
        return loss.mean()

criterion = FocalLoss(alpha=0.75, gamma=2.0, label_smoothing=0.05)
print("Focal Loss criterion with label smoothing configured.")


In [ ]:
def compute_biometric_metrics(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob, pos_label=1)
    fnr = 1.0 - tpr
    abs_diff = np.abs(fpr - fnr)
    min_idx = np.nanargmin(abs_diff)
    eer = float((fpr[min_idx] + fnr[min_idx]) / 2.0)
    opt_thresh = float(np.clip(thresholds[min_idx], 0.0, 1.0))
    auc = float(roc_auc_score(y_true, y_prob))

    c_miss = 1.0
    c_fa = 10.0
    p_tar = 0.05
    p_non = 1.0 - p_tar
    t_dcf_curve = c_miss * p_tar * fnr + c_fa * p_non * fpr
    min_tdcf = float(np.min(t_dcf_curve) / min(c_miss * p_tar, c_fa * p_non))

    return {
        "eer": eer,
        "optimal_threshold": opt_thresh,
        "auc": auc,
        "min_tdcf": min_tdcf,
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds
    }

def run_network_evaluation(net, loader, compute_device, use_mixed_precision):
    net.eval()
    prob_list = []
    target_list = []
    with torch.no_grad():
        for x_b, y_b in loader:
            x_b = x_b.to(compute_device)
            with torch.amp.autocast(device_type=compute_device.type, enabled=use_mixed_precision):
                logits = net(x_b)
                probs = torch.softmax(logits, dim=1)[:, 1]
            prob_list.append(probs.cpu().numpy())
            target_list.append(y_b.numpy())
    all_probs = np.concatenate(prob_list)
    all_targets = np.concatenate(target_list)
    metrics = compute_biometric_metrics(all_targets, all_probs)
    return metrics, all_probs, all_targets

print("Biometric evaluation functions (EER, min t-DCF, ROC) defined.")


In [ ]:
total_epochs = 20
base_lr = 1e-3
min_lr = 1e-6
weight_decay = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=min_lr)
scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)

best_model_path = os.path.join(weights_dir, "se_resnet18_best.pth")
history_file = os.path.join(weights_dir, "training_history.json")

best_dev_eer = float("inf")
best_dev_auc = 0.0
best_dev_threshold = 0.5
training_log = []

print(f"Commencing SE-ResNet-18 End-to-End Training ({total_epochs} Epochs)")
print("=" * 85)

for epoch in range(1, total_epochs + 1):
    t_start = time.time()
    model.train()
    running_train_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_train_loss += loss.item() * x_batch.size(0)

    epoch_loss = running_train_loss / len(train_dataset)
    scheduler.step()

    dev_metrics, _, _ = run_network_evaluation(model, dev_loader, device, use_amp)
    dev_eer = dev_metrics["eer"]
    dev_auc = dev_metrics["auc"]
    dev_thresh = dev_metrics["optimal_threshold"]
    dev_tdcf = dev_metrics["min_tdcf"]

    elapsed = time.time() - t_start
    log_entry = {
        "epoch": epoch,
        "train_loss": round(epoch_loss, 5),
        "dev_eer": round(dev_eer, 5),
        "dev_auc": round(dev_auc, 5),
        "dev_min_tdcf": round(dev_tdcf, 5),
        "dev_threshold": round(dev_thresh, 5),
        "time_seconds": round(elapsed, 1)
    }
    training_log.append(log_entry)

    is_optimal = dev_eer < best_dev_eer
    if is_optimal:
        best_dev_eer = dev_eer
        best_dev_auc = dev_auc
        best_dev_threshold = dev_thresh
        torch.save({
            "epoch": epoch,
            "state_dict": model.state_dict(),
            "eer": best_dev_eer,
            "auc": best_dev_auc,
            "threshold": best_dev_threshold,
            "architecture": "SE-ResNet-18"
        }, best_model_path)

    star = " [NEW BEST]" if is_optimal else ""
    print(f"Epoch [{epoch:02d}/{total_epochs:02d}] | Loss: {epoch_loss:.4f} | Dev EER: {dev_eer*100:.2f}% | Dev AUC: {dev_auc:.4f} | min t-DCF: {dev_tdcf:.4f} | Time: {elapsed:.0f}s{star}")

with open(history_file, "w", encoding="utf-8") as f:
    json.dump(training_log, f, indent=2)

print("=" * 85)
print(f"Training Complete. Optimal Dev EER: {best_dev_eer*100:.2f}% | Dev AUC: {best_dev_auc:.4f}")
print(f"Optimal Checkpoint Saved: {best_model_path}")


In [ ]:
epochs_x = [e["epoch"] for e in training_log]
losses_y = [e["train_loss"] for e in training_log]
eers_y = [e["dev_eer"] * 100 for e in training_log]
aucs_y = [e["dev_auc"] for e in training_log]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_x, losses_y, color="midnightblue", lw=2, marker="o", markersize=4, label="Training Loss (Focal)")
axes[0].set_title("Training Loss Convergence Across Epochs", fontsize=12)
axes[0].set_xlabel("Epoch Number", fontsize=11)
axes[0].set_ylabel("Loss Magnitude", fontsize=11)
axes[0].grid(True, linestyle="--", alpha=0.3)
axes[0].legend(fontsize=10)

axes[1].plot(epochs_x, eers_y, color="crimson", lw=2, marker="s", markersize=4, label="Development EER (%)")
axes[1].plot(epochs_x, [a * 100 for a in aucs_y], color="forestgreen", lw=1.5, linestyle="--", label="Development AUC (x100)")
axes[1].set_title("Biometric Validation Trajectory: Dev EER and AUC", fontsize=12)
axes[1].set_xlabel("Epoch Number", fontsize=11)
axes[1].set_ylabel("Percentage (%)", fontsize=11)
axes[1].grid(True, linestyle="--", alpha=0.3)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "08_training_loss_and_eer_curves.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 08: Training loss and biometric validation trajectory.")


In [ ]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"Loaded optimal checkpoint from epoch {checkpoint['epoch']} with Dev EER: {checkpoint['eer']*100:.2f}%")

dev_metrics, dev_scores, dev_targets = run_network_evaluation(model, dev_loader, device, use_amp)
optimal_threshold = dev_metrics["optimal_threshold"]

print("")
print("Executing Comprehensive Batch Inference on Full ASVspoof 2019 Evaluation Partition...")
print(f"Total Target Evaluation Utterances: {len(eval_df):,}")
t_eval_start = time.time()

eval_metrics, eval_scores, eval_targets = run_network_evaluation(model, eval_loader, device, use_amp)
t_eval_elapsed = time.time() - t_eval_start

print(f"Evaluation Partition Inference Completed in {t_eval_elapsed:.1f}s ({len(eval_df)/max(t_eval_elapsed, 1):.0f} utterances/sec)")
print("")
print("=" * 60)
print("ASVSPOOF 2019 COMPREHENSIVE BIOMETRIC PERFORMANCE BENCHMARK")
print("=" * 60)
print(f"Development Partition (Known Attacks A01 - A06):")
print(f"  EER (%):           {dev_metrics['eer']*100:.3f}%")
print(f"  min t-DCF:         {dev_metrics['min_tdcf']:.4f}")
print(f"  ROC-AUC:           {dev_metrics['auc']:.4f}")
print(f"  Decision Thresh:   {optimal_threshold:.4f}")
print("")
print(f"Evaluation Partition (Unseen Attacks A07 - A19):")
print(f"  EER (%):           {eval_metrics['eer']*100:.3f}%")
print(f"  min t-DCF:         {eval_metrics['min_tdcf']:.4f}")
print(f"  ROC-AUC:           {eval_metrics['auc']:.4f}")
print("=" * 60)


In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(dev_metrics["fpr"], dev_metrics["tpr"], color="steelblue", lw=2, label=f"Dev Partition (AUC = {dev_metrics['auc']:.4f}, EER = {dev_metrics['eer']*100:.2f}%)")
plt.plot(eval_metrics["fpr"], eval_metrics["tpr"], color="crimson", lw=2, label=f"Eval Partition (AUC = {eval_metrics['auc']:.4f}, EER = {eval_metrics['eer']*100:.2f}%)")
plt.plot([0, 1], [0, 1], color="gray", linestyle=":", label="Random Classifier (AUC = 0.5000)")
plt.scatter([dev_metrics["eer"]], [1 - dev_metrics["eer"]], color="blue", s=60, zorder=5, label=f"Dev Operating Point ({dev_metrics['eer']*100:.2f}%)")
plt.scatter([eval_metrics["eer"]], [1 - eval_metrics["eer"]], color="darkred", s=60, zorder=5, label=f"Eval Operating Point ({eval_metrics['eer']*100:.2f}%)")

plt.title("Receiver Operating Characteristic (ROC) Benchmark: Dev vs Eval", fontsize=12)
plt.xlabel("False Positive Rate (FPR)", fontsize=11)
plt.ylabel("True Positive Rate (TPR)", fontsize=11)
plt.legend(loc="lower right", fontsize=9)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "09_receiver_operating_characteristic_roc.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 09: ROC curves (Dev vs Eval).")


In [ ]:
from scipy.stats import norm

def plot_det(fpr, fnr, label, color, ax):
    fpr = np.clip(fpr, 1e-5, 1 - 1e-5)
    fnr = np.clip(fnr, 1e-5, 1 - 1e-5)
    ax.plot(norm.ppf(fpr), norm.ppf(fnr), label=label, color=color, lw=2)

fig, ax = plt.subplots(figsize=(7, 6))
dev_fnr = 1.0 - dev_metrics["tpr"]
eval_fnr = 1.0 - eval_metrics["tpr"]

plot_det(dev_metrics["fpr"], dev_fnr, f"Dev Partition (EER = {dev_metrics['eer']*100:.2f}%)", "steelblue", ax)
plot_det(eval_metrics["fpr"], eval_fnr, f"Eval Partition (EER = {eval_metrics['eer']*100:.2f}%)", "crimson", ax)

ticks = [0.001, 0.01, 0.05, 0.20, 0.50]
tick_locs = norm.ppf(ticks)
tick_labels = [f"{t*100:.1f}%" for t in ticks]
ax.set_xticks(tick_locs)
ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_locs)
ax.set_yticklabels(tick_labels)

ax.set_title("Detection Error Tradeoff (DET) Benchmark: Dev vs Eval", fontsize=12)
ax.set_xlabel("False Alarm Rate / FPR (%)", fontsize=11)
ax.set_ylabel("Miss Rate / FNR (%)", fontsize=11)
ax.legend(loc="upper right", fontsize=10)
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "10_detection_error_tradeoff_det.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 10: DET curves (Dev vs Eval).")


In [ ]:
prec_dev, rec_dev, _ = precision_recall_curve(dev_targets, dev_scores)
prec_eval, rec_eval, _ = precision_recall_curve(eval_targets, eval_scores)
ap_dev = average_precision_score(dev_targets, dev_scores)
ap_eval = average_precision_score(eval_targets, eval_scores)

plt.figure(figsize=(7, 6))
plt.plot(rec_dev, prec_dev, color="steelblue", lw=2, label=f"Dev Partition (AP = {ap_dev:.4f})")
plt.plot(rec_eval, prec_eval, color="crimson", lw=2, label=f"Eval Partition (AP = {ap_eval:.4f})")
plt.title("Precision-Recall (PR) Curve Benchmark: Dev vs Eval", fontsize=12)
plt.xlabel("Recall (True Spoof Coverage)", fontsize=11)
plt.ylabel("Precision (Positive Predictive Value)", fontsize=11)
plt.legend(loc="lower left", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "11_precision_recall_curves.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 11: Precision-Recall curves (Dev vs Eval).")


In [ ]:
eval_preds_binary = (eval_scores >= optimal_threshold).astype(int)
cm = confusion_matrix(eval_targets, eval_preds_binary)
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(6, 5))
sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues", cbar=True,
            xticklabels=["Authentic (Bonafide)", "Synthetic (Spoof)"],
            yticklabels=["Authentic (Bonafide)", "Synthetic (Spoof)"],
            annot_kws={"size": 12, "weight": "bold"})
plt.title(f"Normalized Confusion Matrix: Evaluation Partition (N={len(eval_targets):,})", fontsize=11)
plt.xlabel("Predicted Class Label", fontsize=10)
plt.ylabel("Ground Truth Class Label", fontsize=10)

tn, fp, fn, tp = cm.ravel()
acc_eval = (tp + tn) / len(eval_targets)
prec_eval_val = tp / max(tp + fp, 1)
rec_eval_val = tp / max(tp + fn, 1)
f1_eval_val = 2 * (prec_eval_val * rec_eval_val) / max(prec_eval_val + rec_eval_val, 1e-6)

print(f"Evaluation Confusion Matrix Metrics:")
print(f"  True Negatives (Authentic Correct): {tn:,} ({cm_norm[0,0]*100:.2f}%)")
print(f"  False Positives (Authentic False Alarm): {fp:,} ({cm_norm[0,1]*100:.2f}%)")
print(f"  False Negatives (Spoof Missed): {fn:,} ({cm_norm[1,0]*100:.2f}%)")
print(f"  True Positives (Spoof Detected): {tp:,} ({cm_norm[1,1]*100:.2f}%)")
print(f"  Overall Accuracy:  {acc_eval*100:.2f}%")
print(f"  Overall F1-Score:  {f1_eval_val:.4f}")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "12_normalized_confusion_matrix_eval.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 12: Normalized confusion matrix on the Evaluation partition.")


In [ ]:
eval_df_scored = eval_df.copy()
eval_df_scored["spoof_score"] = eval_scores
eval_df_scored["pred_label"] = eval_preds_binary

attack_descriptions = {
    "A01": "TTS: Neural Acoustic (AR RNN) + WaveNet",
    "A02": "TTS: Neural Acoustic (AR RNN) + WORLD",
    "A03": "TTS: Concatenative Unit Selection",
    "A04": "VC: Formant / Pitch Shifting + STRAIGHT",
    "A05": "VC: Variational Autoencoder (VAE)",
    "A06": "VC: Transfer Function Regression + WORLD",
    "A07": "TTS: Neural Acoustic + Waveform Filtering",
    "A08": "TTS: Neural Acoustic + Spectral Filtering",
    "A09": "TTS: Poly-Phase Vocoder",
    "A10": "TTS: Autoregressive Neural Vocoder (WaveNet)",
    "A11": "TTS: Non-Autoregressive Waveform Synthesis",
    "A12": "TTS: Neural Source-Filter (NSF)",
    "A13": "VC: Differential Formant Synthesis",
    "A14": "VC: Direct Waveform Modification",
    "A15": "VC: Adaptive Waveform Filtering",
    "A16": "VC: Spectral Envelope Transformation",
    "A17": "VC: High-Order Non-Linear Phase Mapping",
    "A18": "VC: Formant-Preserving Pitch Synchronous",
    "A19": "VC: Multi-Speaker Variational Transfer"
}

all_attacks = [f"A{i:02d}" for i in range(1, 20)]
breakdown_records = []

bon_sub = eval_df_scored[eval_df_scored["key"] == "bonafide"]
if len(bon_sub) > 0:
    bon_correct = int((bon_sub["pred_label"] == 0).sum())
    bon_acc = bon_correct / len(bon_sub)
    breakdown_records.append({
        "Attack ID": "Bonafide",
        "Type": "Human Speech",
        "Algorithm Description": "VCTK Authentic Multi-Speaker Audio",
        "Total Utterances": len(bon_sub),
        "Accuracy (%)": round(bon_acc * 100, 2),
        "Mean Spoof Score": round(float(bon_sub["spoof_score"].mean()), 4)
    })

for atk in all_attacks:
    atk_sub = eval_df_scored[eval_df_scored["attack_id"] == atk]
    if len(atk_sub) == 0:
        continue
    correct = int((atk_sub["pred_label"] == 1).sum())
    acc = correct / len(atk_sub)
    mean_sc = float(atk_sub["spoof_score"].mean())
    atk_type = "Known (Train/Dev)" if atk in ["A01","A02","A03","A04","A05","A06"] else "Unseen (Eval OOD)"
    breakdown_records.append({
        "Attack ID": atk,
        "Type": atk_type,
        "Algorithm Description": attack_descriptions.get(atk, "Unspecified Synthesis Architecture"),
        "Total Utterances": len(atk_sub),
        "Accuracy (%)": round(acc * 100, 2),
        "Mean Spoof Score": round(mean_sc, 4)
    })

breakdown_table_df = pd.DataFrame(breakdown_records)
print(breakdown_table_df.to_string(index=False))

breakdown_table_df.to_csv("/kaggle/working/attack_vulnerability_breakdown.csv", index=False)

plt.figure(figsize=(15, 6))
plot_atks = [r for r in breakdown_records if r["Attack ID"] != "Bonafide"]
atk_names = [r["Attack ID"] for r in plot_atks]
atk_accs = [r["Accuracy (%)"] for r in plot_atks]
atk_types = [r["Type"] for r in plot_atks]
colors = ["coral" if "Known" in t else "mediumpurple" for t in atk_types]

bars = plt.bar(atk_names, atk_accs, color=colors, edgecolor="black", width=0.6)
plt.axhline(100.0, color="gray", linestyle=":", lw=1)
plt.axhline(acc_eval * 100, color="crimson", linestyle="--", lw=1.5, label=f"Average Eval Accuracy: {acc_eval*100:.2f}%")
plt.title("Attack-by-Attack Detection Accuracy Breakdown: Known (Coral) vs Unseen (Purple)", fontsize=12)
plt.xlabel("Spoofing Algorithm Identifier (A01 - A19)", fontsize=11)
plt.ylabel("Detection Accuracy (%)", fontsize=11)
plt.ylim(0, 110)
plt.legend(loc="lower right", fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.3)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"{yval:.1f}%", ha="center", va="bottom", fontsize=8, rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "13_attack_by_attack_accuracy_barchart.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 13: Granular attack-by-attack detection accuracy bar chart.")


In [ ]:
bon_eval_sample = eval_df_scored[eval_df_scored["key"] == "bonafide"].sample(min(300, len(eval_df_scored[eval_df_scored["key"] == "bonafide"])), random_state=42)
known_spoofs = eval_df_scored[eval_df_scored["attack_id"].isin(["A04", "A06"])].sample(min(300, len(eval_df_scored[eval_df_scored["attack_id"].isin(["A04", "A06"])])), random_state=42)
unseen_spoofs = eval_df_scored[eval_df_scored["attack_id"].isin(["A07", "A10", "A12", "A17", "A19"])].sample(min(400, len(eval_df_scored[eval_df_scored["attack_id"].isin(["A07", "A10", "A12", "A17", "A19"])])), random_state=42)

tsne_meta = pd.concat([bon_eval_sample, known_spoofs, unseen_spoofs]).reset_index(drop=True)
tsne_ds = ASVSpoofDataset(tsne_meta, is_train=False)
tsne_loader = DataLoader(tsne_ds, batch_size=64, shuffle=False)

extracted_latents = []
model.eval()
with torch.no_grad():
    for x_b, _ in tsne_loader:
        x_b = x_b.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            lat = model.extract_latent(x_b)
        extracted_latents.append(lat.cpu().numpy())

latent_arr = np.concatenate(extracted_latents)
print(f"Latent Embeddings Matrix Extracted: {latent_arr.shape}")

tsne_engine = TSNE(n_components=2, perplexity=35, random_state=42, n_iter=1000)
coords_2d = tsne_engine.fit_transform(latent_arr)

plt.figure(figsize=(9, 7))

is_bon = tsne_meta["key"] == "bonafide"
is_known = tsne_meta["attack_id"].isin(["A04", "A06"])
is_unseen = ~is_bon & ~is_known

plt.scatter(coords_2d[is_bon, 0], coords_2d[is_bon, 1], color="steelblue", alpha=0.8, s=40, label="Authentic Human Voice (Bonafide)")
plt.scatter(coords_2d[is_known, 0], coords_2d[is_known, 1], color="forestgreen", alpha=0.8, s=40, label="Known Attacks (A04, A06)")
plt.scatter(coords_2d[is_unseen, 0], coords_2d[is_unseen, 1], color="crimson", alpha=0.8, s=40, label="Unseen OOD Attacks (A07, A10, A12, A17, A19)")

plt.title("t-SNE 2D Projection of SE-ResNet-18 128-Dimensional Latent Manifold", fontsize=12)
plt.xlabel("t-SNE Dimension 1", fontsize=11)
plt.ylabel("t-SNE Dimension 2", fontsize=11)
plt.legend(loc="best", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "14_tsne_latent_manifold_clusters.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 14: t-SNE 2D latent manifold clustering.")


In [ ]:
class GradCAM:
    def __init__(self, target_model, target_layer):
        self.model = target_model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, inp, out):
            self.activations = out.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        h1 = self.target_layer.register_forward_hook(forward_hook)
        h2 = self.target_layer.register_full_backward_hook(backward_hook)
        self.hook_handles.extend([h1, h2])

    def generate_heatmap(self, raw_audio_tensor, target_class=1):
        self.model.eval()
        self.model.zero_grad()

        raw_audio_tensor = raw_audio_tensor.to(device)
        with torch.enable_grad():
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                logits = self.model(raw_audio_tensor)
                score = logits[0, target_class]
            score.backward()

        grads = self.gradients[0]
        acts = self.activations[0]
        weights = torch.mean(grads, dim=[1, 2], keepdim=True)
        cam = torch.sum(weights * acts, dim=0)
        cam = F.relu(cam)
        cam = cam - cam.min()
        if cam.max() > 1e-6:
            cam = cam / cam.max()
        return cam.cpu().numpy()

    def remove_hooks(self):
        for h in self.hook_handles:
            h.remove()

grad_cam_engine = GradCAM(model, model.layer4[-1])

bon_test_file = eval_df_scored[eval_df_scored["key"] == "bonafide"].iloc[0]["file_path"]
spf_test_file = eval_df_scored[eval_df_scored["attack_id"] == "A10"].iloc[0]["file_path"]

fig, axes = plt.subplots(2, 2, figsize=(15, 7))

for idx, (fpath, title_prefix, class_target) in enumerate([
    (bon_test_file, "Authentic Speech (Bonafide)", 0),
    (spf_test_file, "Synthetic Speech (WaveNet A10)", 1)
]):
    raw_sig, _ = read_audio_file(fpath)
    norm_sig = normalize_waveform(raw_sig, target_len=64000)
    audio_t = torch.from_numpy(norm_sig).float().unsqueeze(0)

    mel_t = model.extract_features(audio_t.to(device)).squeeze(0).squeeze(0).cpu().numpy()
    heatmap = grad_cam_engine.generate_heatmap(audio_t, target_class=class_target)

    from scipy.ndimage import zoom
    zoom_factors = (mel_t.shape[0] / heatmap.shape[0], mel_t.shape[1] / heatmap.shape[1])
    resized_heatmap = zoom(heatmap, zoom_factors, order=1)

    axes[idx, 0].imshow(mel_t, origin="lower", aspect="auto", cmap="viridis")
    axes[idx, 0].set_title(f"{title_prefix}: Normalized Log-Mel Spectrogram", fontsize=10)
    axes[idx, 0].set_ylabel("Mel Frequency Bins", fontsize=9)

    axes[idx, 1].imshow(mel_t, origin="lower", aspect="auto", cmap="gray")
    im_cam = axes[idx, 1].imshow(resized_heatmap, origin="lower", aspect="auto", cmap="jet", alpha=0.55)
    axes[idx, 1].set_title(f"{title_prefix}: Grad-CAM Saliency Attribution", fontsize=10)
    axes[idx, 1].set_ylabel("Mel Frequency Bins", fontsize=9)
    fig.colorbar(im_cam, ax=axes[idx, 1], fraction=0.046, pad=0.04)

axes[1, 0].set_xlabel("Time Frame Index", fontsize=10)
axes[1, 1].set_xlabel("Time Frame Index", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "15_gradcam_spectro_temporal_explainability.png"), dpi=300, bbox_inches="tight")
plt.show()
grad_cam_engine.remove_hooks()
print("Saved Figure 15: Grad-CAM spectro-temporal explainability heatmaps.")


In [ ]:
def predict_single_file(file_path, net, target_threshold):
    net.eval()
    sig, _ = read_audio_file(file_path)
    sig_proc = normalize_waveform(sig, target_len=64000)
    tensor_input = torch.from_numpy(sig_proc).float().unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            prob_spoof = torch.softmax(net(tensor_input), dim=1)[0, 1].item()

    is_detected_spoof = prob_spoof >= target_threshold
    decision = "SPOOF (SYNTHETIC VOICE DETECTED)" if is_detected_spoof else "BONAFIDE (AUTHENTIC HUMAN VOICE)"
    confidence = prob_spoof if is_detected_spoof else (1.0 - prob_spoof)

    return {
        "file_name": os.path.basename(file_path),
        "decision": decision,
        "spoof_probability": round(prob_spoof, 5),
        "confidence": f"{confidence * 100:.2f}%",
        "operating_threshold": round(target_threshold, 4)
    }

demo_bon = eval_df_scored[eval_df_scored["key"] == "bonafide"].iloc[1]["file_path"]
demo_spf = eval_df_scored[eval_df_scored["attack_id"] == "A12"].iloc[1]["file_path"]

print("Live File-Level Biometric Inference Demonstration:")
print("")
print("--- Test Sample 1: Ground Truth Authentic ---")
res_bon = predict_single_file(demo_bon, model, optimal_threshold)
print(json.dumps(res_bon, indent=2))

print("")
print("--- Test Sample 2: Ground Truth Deepfake (A12 Neural Source-Filter) ---")
res_spf = predict_single_file(demo_spf, model, optimal_threshold)
print(json.dumps(res_spf, indent=2))

fig, ax = plt.subplots(figsize=(8, 3))
test_names = ["Sample 1 (Authentic)", "Sample 2 (NSF Deepfake A12)"]
test_probs = [res_bon["spoof_probability"], res_spf["spoof_probability"]]
test_colors = ["steelblue", "crimson"]

bars = ax.barh(test_names, test_probs, color=test_colors, height=0.4)
ax.axvline(optimal_threshold, color="black", linestyle="--", lw=1.5, label=f"Decision Threshold ({optimal_threshold:.4f})")
ax.set_xlim(0, 1.0)
ax.set_xlabel("Spoof Posterior Probability", fontsize=11)
ax.set_title("Single-File Live Inference Confidence Benchmark", fontsize=12)
ax.legend(loc="lower right", fontsize=10)
ax.grid(axis="x", linestyle="--", alpha=0.3)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.02, bar.get_y() + bar.get_height()/2.0, f"{w:.4f}", va="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "16_single_file_inference_verification.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved Figure 16: Single-file live inference verification.")


In [ ]:
final_manifest = {
    "study_metadata": {
        "architecture": "SE-ResNet-18 (Squeeze-and-Excitation)",
        "front_end": "80-bin Log-Mel Spectrogram (GPU Transform)",
        "dataset": "ASVspoof 2019 Logical Access",
        "training_epochs": total_epochs,
        "batch_size": batch_size,
        "loss_function": "Focal Loss (alpha=0.75, gamma=2.0, smoothing=0.05)"
    },
    "development_metrics": {
        "eer_percent": round(dev_metrics["eer"] * 100, 3),
        "min_tdcf": round(dev_metrics["min_tdcf"], 4),
        "auc": round(dev_metrics["auc"], 4),
        "optimal_threshold": round(optimal_threshold, 4)
    },
    "evaluation_metrics": {
        "eer_percent": round(eval_metrics["eer"] * 100, 3),
        "min_tdcf": round(eval_metrics["min_tdcf"], 4),
        "auc": round(eval_metrics["auc"], 4),
        "overall_accuracy_percent": round(acc_eval * 100, 2),
        "f1_score": round(f1_eval_val, 4),
        "total_eval_utterances": len(eval_targets)
    }
}

manifest_output_path = "/kaggle/working/experiment_final_report.json"
with open(manifest_output_path, "w", encoding="utf-8") as f:
    json.dump(final_manifest, f, indent=2)

print("")
print("=" * 80)
print("FINAL ARTIFACT INVENTORY AND VERIFICATION")
print("=" * 80)
print(f"1. Checkpoint:        {best_model_path}")
print(f"2. Training History:  {history_file}")
print(f"3. Final Report JSON: {manifest_output_path}")
print(f"4. Attack Breakdown:  /kaggle/working/attack_vulnerability_breakdown.csv")
print("5. Diagnostic Figures in /kaggle/working/figures/:")
for fig_file in sorted(os.listdir(fig_dir)):
    sz = os.path.getsize(os.path.join(fig_dir, fig_file))
    print(f"   - {fig_file} ({sz/1024:.1f} KB)")
print("=" * 80)
print("Research experiment run completed successfully.")
